In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import sys
#sys.path.insert('/Users/mguelfan/Documents/GRAND/ADF_DC2/ADFRecons/PTREND/')
from recons_Xmax_truedirection_omegac_simus import *


font = { 'weight' : 'normal', 'size'   : 20}
plt.rc('font', **font)
plt.rcParams['figure.figsize'] = (15, 10)
plt.rc('legend',fontsize=20)
parameters = {'axes.labelsize': 20}
plt.rcParams.update(parameters)

TypeError: insert expected 2 arguments, got 1

In [6]:
B_dec = 0.
B_inc = np.pi/2. + 1.0609856522873529
groundAltitude = 1086.0
c_light = 2.997924580e8
Bvec = np.array([np.sin(B_inc)*np.cos(B_dec),np.sin(B_inc)*np.sin(B_dec),np.cos(B_inc)])
R_earth = 6371007.0
ns = 325
kr = -0.1218


In [7]:
def event_name_random(dataframe):
    #Obtenir les valeurs uniques de la colonne 'event name'
    valeurs_event_name = dataframe['EventName'].unique()
    eventname = random.choice(valeurs_event_name)
    return eventname

def filter_data_by_antenna_position(dataframe, eta_pos, eta_neg, error=1):
    """
    Filter a DataFrame based on antenna positions.
    
    Args:
    dataframe (DataFrame): The DataFrame containing the data to filter.
    eta_pos (float): Position of the antennas on the side of positive vxvxB (including antennas with eta = 0°).
    eta_neg (float): Position of the antennas on the side of negative vxvxB (including antennas with eta = 180°).
    error (float, optional): The error margin for antenna position. Default is 1.
    
    Returns:
    DataFrame: The filtered DataFrame for both antenna positions.
    """
    # Create masks for both antenna positions
    mask_pos = (dataframe['eta'] >= eta_pos - error) & (dataframe['eta'] <= eta_pos + error)

    if eta_neg == 180:
        mask_neg = (
    (dataframe['eta'] >= (eta_neg - error)) & (dataframe['eta'] <= (eta_neg + error)) |  
    (dataframe['eta'] >= (-eta_neg - error)) & (dataframe['eta'] <= (-eta_neg + error))  

)
    else:
        mask_neg = (dataframe['eta'] >= eta_neg - error) & (dataframe['eta'] <= eta_neg + error)  

    # Apply masks to filter the DataFrame
    dataframe_filtered_pos = dataframe[mask_pos]
    dataframe_filtered_neg = dataframe[mask_neg]

    return dataframe_filtered_pos, dataframe_filtered_neg

In [8]:
def ZHSEffectiveRefractionIndex(X0,Xa):

    R02 = X0[0]**2 + X0[1]**2
    
    # Altitude of emission in km
    h0 = (np.sqrt( (X0[2]+R_earth)**2 + R02 ) - R_earth)/1e3
    # print('Altitude of emission in km = ',h0)
    # print(h0)
    
    # Refractivity at emission 
    rh0 = ns*np.exp(kr*h0)

    modr = np.sqrt(R02)
    # print(modr)

    if (modr > 1e3):

        # Vector between antenna and emission point
        U = Xa-X0
        # Divide into pieces shorter than 10km
        #nint = np.int(modr/2e4)+1
        nint = int(modr/2e4)+1
        K = U/nint

        # Current point coordinates and altitude
        Curr  = X0
        currh = h0
        s = 0.

        for i in np.arange(nint):
            Next = Curr + K # Next point
            nextR2 = Next[0]*Next[0] + Next[1]*Next[1]
            nexth  = (np.sqrt( (Next[2]+R_earth)**2 + nextR2 ) - R_earth)/1e3
            if (np.abs(nexth-currh) > 1e-10):
                s += (np.exp(kr*nexth)-np.exp(kr*currh))/(kr*(nexth-currh))
            else:
                s += np.exp(kr*currh)

            Curr = Next
            currh = nexth
            # print (currh)

        avn = ns*s/nint
        # print(avn)
        n_eff = 1. + 1e-6*avn # Effective (average) index

    else:

        # without numerical integration
        hd = Xa[2]/1e3 # Antenna altitude
        #if (np.abs(hd-h0) > 1e-10):
        avn = (ns/(kr*(hd-h0)))*(np.exp(kr*hd)-np.exp(kr*h0))
        #else:
        #    avn = ns*np.exp(kr*h0)

        n_eff = 1. + 1e-6*avn # Effective (average) index

    return (n_eff)

def ADF_3D_parameters_before_after_Xmax(params, Aants, Xants, Xmax, omega_cerenkov_array, asym_coeff=0.01):
    
    '''

    Computes amplitude prediction for each antenna (i):
    residuals[i] = f_i^{ADF}(\theta,\phi,\delta\omega,A,r_xmax)
    where the ADF function reads:
    
    f_i = f_i(\omega_i, \eta_i, \alpha, l_i, \delta_omega, A)
        = A/l_i f_geom(\alpha, \eta_i) f_Cerenkov(\omega,\delta_\omega)
    
    where 
    
    f_geom(\alpha, \eta_i) = (1 + B \sin(\alpha))**2 \cos(\eta_i) # B is here the geomagnetic asymmetry
    f_Cerenkov(\omega_i,\delta_\omega) = 1 / (1+4{ (\tan(\omega_i)/\tan(\omega_c))**2 - 1 ) / \delta_\omega }**2 )
    
    Input parameters are: params = theta, phi, delta_omega, amplitude
    \theta, \phi define the shower direction angles, \delta_\omega the width of the Cerenkov ring, 
    A is the amplitude paramater, r_xmax is the norm of the position vector at Xmax.

    Derived parameters are: 
    \alpha, angle between the shower axis and the magnetic field
    \eta_i is the azimuthal angle of the (projection of the) antenna position in shower plane
    \omega_i is the angle between the shower axis and the vector going from Xmax to the antenna position

    '''

    theta, phi, delta_omega, amplitude = params
    nants = Xants.shape[0]
    ct = np.cos(theta); st = np.sin(theta); cp = np.cos(phi); sp = np.sin(phi)
    # Define shower basis vectors
    K = np.array([st*cp,st*sp,ct])
    K_plan = np.array([K[0],K[1]])
    KxB = np.cross(K,Bvec); KxB /= np.linalg.norm(KxB)
    KxKxB = np.cross(K,KxB); KxKxB /= np.linalg.norm(KxKxB)
    # Coordinate transform matrix
    mat = np.vstack((KxB,KxKxB,K))
    # 
    XmaxDist = (groundAltitude-Xmax[2])/K[2]
    # print('XmaxDist = ',XmaxDist)
    asym = asym_coeff * (1. - np.dot(K,Bvec)**2) # Azimuthal dependence, in \sin^2(\alpha)
    #

    # Loop on antennas. Here no precomputation table is possible for Cerenkov angle computation.
    # Calculation needs to be done for each antenna.
    res = np.zeros(nants)
    eta_array = np.zeros(nants)
    omega_array = np.zeros(nants)
    omega_cerenkov_simu = np.zeros(nants)

    for i in range(nants):
        # Antenna position from Xmax
        dX = Xants[i,:]-Xmax
        # Expressed in shower frame coordinates
        dX_sp = np.dot(mat,dX)
        #
        l_ant = np.linalg.norm(dX)
        eta = np.arctan2(dX_sp[1],dX_sp[0])
        omega = np.arccos(np.dot(K,dX)/l_ant)

        eta = np.array([np.rad2deg(eta)])
        bins = np.array([-190, -160, -110, -70, -20, 20, 70, 110, 160, 190])
        #eta is contained inside the edges of one of the bins 
        #eta takes the value of the label corresponding to the bin
        #eg: eta = -115: eta is inside the bin [-160, -110[, and takes the value label=-135
        bins_etas = pd.cut(eta, bins, labels=np.array([180, -135, -90, -45, 0, 45, 90, 135, 180]), ordered=False)
        #find the index corresponding to eta_array, and find the associated omega_cr value
        eta_label = np.array([-135, -90, -45, 0, 45, 90, 135, 180])
        index = np.where(eta_label == bins_etas[0])
        index = np.int64(index[0][0])  
        omega_cr = np.deg2rad(omega_cerenkov_array[index])
        eta = np.deg2rad(eta)
        # print ("omega_cr = ",omega_cr)
        # Distribution width. Here rescaled by ratio of cosines (why ?)
        width = ct / (dX[2]/l_ant) * delta_omega
        # Distribution
        adf = amplitude/l_ant / (1.+4.*( ((np.tan(omega)/np.tan(omega_cr))**2 - 1. )/width )**2)
        adf *= 1. + asym*np.cos(eta) # 
        # Chi2
        res[i]= (Aants[i]-adf)
        eta_array[i] = eta
        omega_array[i] = omega
        omega_cerenkov_simu[i]= omega_cr

    return(eta_array, omega_array, omega_cerenkov_simu)

In [ ]:
output_directory = '/Users/mguelfan/Documents/GRAND/ADF_DC2/output_recons_starshape/test_shape/'
an = antenna_set(f'{output_directory}coord_antennas.txt')
an.coordinates -= an.coordinates.mean(axis=0)
co = coincidence_set(f'{output_directory}Rec_coinctable.txt',an)

In [ ]:
file_Cerenkov_asym_Xrecons = '/Users/mguelfan/Documents/GRAND/ADF_DC2/output_recons_starshape/test_frequency/30_100/Rec_adf_amplitude_residuals_3D_before_after_Xmax.txt'
dataframe_Cerenkov_asym_Xrecons = pd.read_csv(file_Cerenkov_asym_Xrecons, sep='\t', dtype={'EventName': float}, names = ['EventName', 'NumberAntennas', 'AmpSimu', 'Amprecons', 'eta', 'omega', 'omega_cr',  'omega_cr_analytic', 'omega_cr_analytic_effectif', 'PosGroundX', 'PosGroundY', 'PosGroundZ'])
dataframe_Cerenkov_asym_Xrecons['EventName'] = dataframe_Cerenkov_asym_Xrecons['EventName'].astype(float).apply('{:.0f}'.format)

input_simu_Cerenkov_asym_Xrecons = '/Users/mguelfan/Documents/GRAND/ADF_DC2/output_recons_starshape/test_frequency/30_100/adf_recons_stats_before_after_Xmax.txt'
simu_dataframe_Cerenkov_asym_Xrecons = pd.read_csv(input_simu_Cerenkov_asym_Xrecons, sep='\s+', dtype={'EventName': float} , names = ['EventName', 'Azimuth', 'Zenith', 'Energy', 'Primary', 'XmaxDistance', 'SlantXmax', 'x_Xmax', 'y_Xmax', 'z_Xmax', 'AntennasNumber', 'ZenithRec', 'AzimuthRec', 'Chi2', 'WidthRec', 'AmpRec', 'AzimErrors', 'ZenErrors', 'AngularDistances', 'adf_time'])
simu_dataframe_Cerenkov_asym_Xrecons['EventName'] = simu_dataframe_Cerenkov_asym_Xrecons['EventName'].astype(float).apply('{:.0f}'.format)
simu_dataframe_Cerenkov_asym_Xrecons = simu_dataframe_Cerenkov_asym_Xrecons.loc[(simu_dataframe_Cerenkov_asym_Xrecons['Energy'] == 3.9811)]

#simu_dataframe_Cerenkov_asym_Xrecons = simu_dataframe_Cerenkov_asym_Xrecons.loc[(simu_dataframe_Cerenkov_asym_Xrecons['Azimuth'] == 180) | (simu_dataframe_Cerenkov_asym_Xrecons['Azimuth'] == 0)]
simu_dataframe_Cerenkov_asym_Xrecons = simu_dataframe_Cerenkov_asym_Xrecons.loc[(simu_dataframe_Cerenkov_asym_Xrecons['Energy'] == 3.9811)]
merged_df_Xrecons = pd.merge(dataframe_Cerenkov_asym_Xrecons, simu_dataframe_Cerenkov_asym_Xrecons, on='EventName')
pd.set_option('display.max_columns', None)
merged_df_Xrecons.dropna(inplace=True)

EventName_1 = '18701803981'
merged_df_Xrecons = merged_df_Xrecons[merged_df_Xrecons['EventName'] == EventName_1]
print(merged_df_Xrecons)

dataframe_filtre_0, dataframe_filtre_180  = filter_data_by_antenna_position(merged_df_Xrecons, 0, 180, error=20)
dataframe_filtre_90, dataframe_filtre_min90  = filter_data_by_antenna_position(merged_df_Xrecons, 90, -90, error=20)
dataframe_filtre_45, dataframe_filtre_min135  = filter_data_by_antenna_position(merged_df_Xrecons, 45, -135, error=20)
dataframe_filtre_135, dataframe_filtre_min45  = filter_data_by_antenna_position(merged_df_Xrecons, 135, -45, error=20)

